###Section D

In [0]:
def id_validations(id):
    value = str(id).strip().upper()
    return None if value == "" else value

In [0]:
dbutils.widgets.text("catalog", "vp_ms20010501")
catalog = dbutils.widgets.get("catalog")

###D1

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DecimalType, IntegerType, DateType
from pyspark.sql.functions import udf

validate_id = udf(id_validations, StringType())

stations_df = (
    spark.read.table(f"{catalog}.bronze.stations")
    .select(
        validate_id(F.col("station_id")).alias("station_id"),
        F.trim(F.col("station_name")).alias("station_name"),
        F.initcap(F.trim(F.col("city"))).alias("city"),
        F.upper(F.trim(F.col("site_type"))).alias("site_type"),
        F.to_date(F.col("opened_on")).cast(DateType()).alias("opened_on"),
    )
)

stations_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stations")

tariffs_df = (
    spark.read.table(f"{catalog}.bronze.tariffs")
    .select(
        F.col("connector_type"),
        F.upper(F.trim(F.col("time_band"))).alias("time_band"),
        F.col("rate_per_kwh").cast(DecimalType(10, 2)).alias("rate_per_kwh"),
        F.col("idle_fee_per_min").cast(DecimalType(10, 2)).alias("idle_fee_per_min"),
        F.col("free_idle_minutes").cast(IntegerType()).alias("free_idle_minutes"),
    )
)

tariffs_df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.tariffs")

###D2

In [0]:
from pyspark.sql.types import BooleanType, DateType, DecimalType
from pyspark.sql.window import Window

target_table = f"{catalog}.silver.chargers"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        charger_id STRING,
        station_id STRING,
        connector_type STRING,
        max_kw DECIMAL(6,1),
        is_active BOOLEAN,
        commissioned_on DATE,
        last_changed_snapshot_date DATE
    )
""")
spark.sql(f"ALTER TABLE {target_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

charger_df = (
    spark.read.table(f"{catalog}.bronze.chargers")
    .select(
        F.col("charger_id"),
        validate_id(F.col("station_id")).alias("station_id"),
        F.upper(F.col("connector.type")).alias("connector_type"),
        F.col("connector.max_kw").cast(DecimalType(6, 1)).alias("max_kw"),
        F.col("is_active").cast(BooleanType()).alias("is_active"),
        F.col("commissioned_on").cast(DateType()).alias("commissioned_on"),
        F.to_date(F.col("snapshot_date")).cast(DateType()).alias("last_changed_snapshot_date"),
    )
)

window = Window.partitionBy("charger_id").orderBy(F.col("last_changed_snapshot_date").desc())
latest_df = (
    charger_df
    .withColumn("row_num", F.row_number().over(window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

latest_df.createOrReplaceTempView("chargers_latest")

spark.sql(f"""
    MERGE INTO {target_table} AS t
    USING chargers_latest AS s
    ON t.charger_id = s.charger_id
    WHEN MATCHED AND (
           NOT (t.station_id       <=> s.station_id)
        OR NOT (t.connector_type   <=> s.connector_type)
        OR NOT (t.max_kw           <=> s.max_kw)
        OR NOT (t.is_active        <=> s.is_active)
        OR NOT (t.commissioned_on  <=> s.commissioned_on)
    )
    THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

###D3

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DateType

# 1. Read and standardize Bronze data
bronze_df = spark.read.table(f"{catalog}.bronze.members")

std_df = bronze_df.select(
    validate_id(F.col('member_id')).alias('member_id'),
    F.trim(F.col('full_name')).alias('full_name'),
    F.lower(F.trim(F.col('email'))).alias('email'),
    F.regexp_replace(F.col('phone'), '[^0-9]', '').substr(-10, 10).alias('phone'),
    F.initcap(F.trim(F.col('city'))).alias('city'),
    F.upper(F.trim(F.col('plan'))).alias('plan'),
    F.when(F.col('joined_on').contains('-'), F.try_to_date(F.col('joined_on'), 'yyyy-MM-dd'))
     .otherwise(F.try_to_date(F.col('joined_on'), 'dd/MM/yyyy')).alias('joined_on'),
    F.col('change_effective_date').cast(DateType())
)

snapshot_date = F.lit('2026-04-01').cast(DateType())

# 2. Initial Snapshot (joined_on <= snapshot_date)
initial_df = std_df.filter(F.col('joined_on') <= snapshot_date) \
    .dropDuplicates(['member_id', 'full_name', 'email', 'phone', 'city', 'plan', 'joined_on']) \
    .withColumn('effective_from', snapshot_date) \
    .withColumn('event_date', snapshot_date)

# 3. Change Rows / Subsequent Events
changes_df = std_df.filter(
    (F.col('joined_on') > snapshot_date) | 
    (F.col('change_effective_date') > snapshot_date)
).withColumn('event_date', F.col('change_effective_date'))

# Unify initial snapshot and change events chronologically per member
initial_events = initial_df.select(
    'member_id', 'full_name', 'email', 'phone', 'city', 'plan', 'joined_on', 'event_date'
)
change_events = changes_df.select(
    'member_id', 'full_name', 'email', 'phone', 'city', 'plan', 'joined_on', 'event_date'
)

all_events = initial_events.unionByName(change_events)

# 4. SCD2 Versioning Logic (Rules 3 & 4)
member_window = Window.partitionBy('member_id').orderBy('event_date', F.col('joined_on'))

# Detect structural changes (plan or city) that trigger a new version
all_events = all_events.withColumn('prev_city', F.lag('city').over(member_window)) \
                       .withColumn('prev_plan', F.lag('plan').over(member_window))

all_events = all_events.withColumn(
    'is_scd2_change',
    F.col('prev_city').isNull() | 
    (F.col('city') != F.col('prev_city')) | 
    (F.col('plan') != F.col('prev_plan'))
)

# Assign a cumulative version sequence number for each member
all_events = all_events.withColumn(
    'version_seq',
    F.sum(F.col('is_scd2_change').cast('int')).over(member_window)
)

# 5. Handle Overwrites within Versions (Rule 4)
# Within the same version_seq, take the latest demographic info (full_name, email, phone)
version_window = Window.partitionBy('member_id', 'version_seq').orderBy('event_date')

versioned_df = all_events.select(
    'member_id',
    'plan',
    'city',
    'joined_on',
    'version_seq',
    F.first('event_date').over(version_window).alias('effective_from'),
    F.last('full_name').over(version_window).alias('full_name'),
    F.last('email').over(version_window).alias('email'),
    F.last('phone').over(version_window).alias('phone')
).dropDuplicates(['member_id', 'version_seq'])

# 6. Final SCD2 Effective Dates & Status
scd2_window = Window.partitionBy('member_id').orderBy('effective_from')

final_members = versioned_df.withColumn(
    'effective_to',
    F.lead('effective_from').over(scd2_window)
).withColumn(
    'is_current',
    F.col('effective_to').isNull()
)

# 7. Write to Silver Table
(final_members
    .select('member_id', 'full_name', 'email', 'phone', 'city', 'plan', 'joined_on', 'effective_from', 'effective_to', 'is_current')
    .write.mode('overwrite')
    .saveAsTable(f"{catalog}.silver.members"))